In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder

# =========================
# 1. 데이터 불러오기
# =========================

df1 = pd.read_csv("서울실거래가_2022.csv", encoding="cp949", skiprows=15)
df2 = pd.read_csv("서울실거래가_2023.csv", encoding="cp949", skiprows=15)
df3 = pd.read_csv("서울실거래가_2024.csv", encoding="cp949", skiprows=15)
df4 = pd.read_csv("서울실거래가_2025.csv", encoding="cp949", skiprows=15)

df = pd.concat([df1, df2, df3, df4])

# =========================
# 2. 필요한 컬럼 선택
# =========================

df = df[['시군구', '전용면적(㎡)', '계약년월', '층', '거래금액(만원)', '건축년도']]

# =========================
# 3. 전처리
# =========================

df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)
df['연도'] = df['계약년월'] // 100
df = df.dropna()

# =========================
# 4. 컬럼명 변경
# =========================

df = df.rename(columns={
    '시군구': 'region',
    '전용면적(㎡)': 'area',
    '계약년월': 'contract_date',
    '층': 'floor',
    '거래금액(만원)': 'price',
    '건축년도': 'year_built',
    '연도': 'year'
})

# =========================
# 5. 지역 인코딩
# =========================

le = LabelEncoder()
df['region_encoded'] = le.fit_transform(df['region'])

# =========================
# 6. 모델 데이터 구성
# =========================

X = df[['area', 'floor', 'year_built', 'year', 'region_encoded']]
y = df['price']

# =========================
# 7. 교차 검증
# =========================

model = XGBRegressor()

scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print("평균 R2:", scores.mean())

# =========================
# 8. 학습/테스트 분리
# =========================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model.fit(X_train, y_train)

# =========================
# 9. 예측
# =========================

y_pred = model.predict(X_test)

# =========================
# 10. 이상 거래 탐지
# =========================

residual = y_test - y_pred

threshold = np.mean(abs(residual)) + 2 * np.std(residual)

anomaly = abs(residual) > threshold

anomaly_df = X_test[anomaly].copy()
anomaly_df['actual_price'] = y_test[anomaly]
anomaly_df['predicted_price'] = y_pred[anomaly]

print("이상 거래 개수:", len(anomaly_df))
print(anomaly_df.head(10))

# =========================
# 11. 그래프 (Scatter)
# =========================

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.scatter(y_test[anomaly], y_pred[anomaly], color='red')

plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red')

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Price")
plt.show()

# =========================
# 12. Residual 분포 (추가)
# =========================

plt.hist(residual, bins=50)
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.show()

# =========================
# 13. Feature Importance
# =========================

importance = model.feature_importances_
features = X.columns

plt.barh(features, importance)
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.show()